In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd

In [ ]:
tickers = [
    "SPY", "IVV", "VOO",
    "QQQ", "VTI", "IWM",
    "XLK", "SOXX"
]

In [ ]:
prices = yf.download(
    tickers,
    period = "1y",
    interval = "1d",
    auto_adjust=True,
    progress = False,
)["Close"]
prices = prices.dropna()
returns = prices.pct_change().dropna()

# Σ = BΩBᵀ + D equation

# Estimate mu

In [ ]:
# Estimate annualised expected returns
mu = R.mean() * 252

print("Expected returns (mu):")
print(mu.round(4))

Expected returns (mu):
SPY     0.1737
IVV     0.1746
VOO     0.1741
QQQ     0.2490
VTI     0.1707
IWM     0.1845
XLK     0.3853
SOXX    0.8742
dtype: float64


# Regress returns on the factors

In [ ]:
# Factor loadings B
B = pd.DataFrame(
    {
        "MKT": [
            R[ticker].cov(F["MKT"]) / F["MKT"].var()
            for ticker in tickers
        ]
    },
    index=tickers
)

# Factor covariance
Omega = F.cov()

# Regression residuals
residuals = R - F.values @ B.values.T

# Idiosyncratic covariance
D = residuals.cov()

print("B:")
print(B.round(3))

print("\nOmega:")
print(Omega.round(6))

print("\nD:")
print(D.round(6))

B:
        MKT
SPY   0.997
IVV   0.996
VOO   0.993
QQQ   1.421
VTI   1.012
IWM   1.194
XLK   1.739
SOXX  2.602

Omega:
          MKT
MKT  0.000067

D:
           SPY       IVV       VOO       QQQ       VTI       IWM       XLK  \
SPY   0.000000  0.000000  0.000000  0.000001  0.000000 -0.000001  0.000001   
IVV   0.000000  0.000001  0.000000  0.000000  0.000000 -0.000001  0.000000   
VOO   0.000000  0.000000  0.000000  0.000001  0.000000 -0.000001  0.000001   
QQQ   0.000001  0.000000  0.000001  0.000023 -0.000000 -0.000004  0.000036   
VTI   0.000000  0.000000  0.000000 -0.000000  0.000000  0.000003  0.000000   
IWM  -0.000001 -0.000001 -0.000001 -0.000004  0.000003  0.000043 -0.000003   
XLK   0.000001  0.000000  0.000001  0.000036  0.000000 -0.000003  0.000077   
SOXX  0.000002  0.000001  0.000001  0.000075  0.000002  0.000023  0.000139   

          SOXX  
SPY   0.000002  
IVV   0.000001  
VOO   0.000001  
QQQ   0.000075  
VTI   0.000002  
IWM   0.000023  
XLK   0.000139  
SOXX  0.00

assemble Σ = BΩBᵀ + D

In [ ]:
Sigma = 252 * (
    B.values @ Omega.values @ B.values.T
    + D.values
)

Sigma = pd.DataFrame(
    Sigma,
    index=tickers,
    columns=tickers
)

print("Sigma = BΩBᵀ + D:")
print(Sigma.round(6))

Sigma = BΩBᵀ + D:
           SPY       IVV       VOO       QQQ       VTI       IWM       XLK  \
SPY   0.016897  0.016895  0.016804  0.024111  0.017076  0.019958  0.029448   
IVV   0.016895  0.016921  0.016803  0.024035  0.017063  0.019878  0.029318   
VOO   0.016804  0.016803  0.016726  0.023983  0.016998  0.019864  0.029320   
QQQ   0.024111  0.024035  0.023983  0.039856  0.024271  0.027570  0.050793   
VTI   0.017076  0.017063  0.016998  0.024271  0.017418  0.021271  0.029744   
IWM   0.019958  0.019878  0.019864  0.027570  0.021271  0.034904  0.034390   
XLK   0.029448  0.029318  0.029320  0.050793  0.029744  0.034390  0.070510   
SOXX  0.044235  0.043923  0.043950  0.081343  0.045008  0.058397  0.111365   

          SOXX  
SPY   0.044235  
IVV   0.043923  
VOO   0.043950  
QQQ   0.081343  
VTI   0.045008  
IWM   0.058397  
XLK   0.111365  
SOXX  0.215218  


# Run naive MVO

In [ ]:
# Naive MVO
mu = R.mean().values * 252
Sigma_np = Sigma.values
n = len(tickers)

# KKT system
KKT = np.block([
    [lam * Sigma_np, np.ones((n, 1))],
    [np.ones((1, n)), np.zeros((1, 1))]
])

rhs = np.concatenate([
    mu,
    [1.0]
])

solution = np.linalg.solve(KKT, rhs)

w_star = solution[:n]

print("Naive MVO weights:")
print(pd.Series(w_star, index=tickers).round(4))

print("\nSum of weights:", round(w_star.sum(), 6))
print("Gross exposure:", round(np.abs(w_star).sum(), 6))

Naive MVO weights:
SPY    -27.9142
IVV      8.2189
VOO     39.9820
QQQ    -12.2952
VTI     -9.8008
IWM     -1.9377
XLK      1.4479
SOXX     3.2989
dtype: float64

Sum of weights: 1.0
Gross exposure: 104.895681


# Nudge μ by 5 basis points

In [ ]:
# Small expected-return error: +5 basis points to SPY
mu_nudged = mu.copy()
mu_nudged[0] += 0.0005

# Solve fully-invested MVO with nudged expected returns
rhs_nudged = np.concatenate([
    mu_nudged,
    [1.0]
])

solution_nudged = np.linalg.solve(
    KKT,
    rhs_nudged
)

w_nudged = solution_nudged[:n]

# Weight movement
weight_change = w_nudged - w_star

print("Original weights:")
print(pd.Series(w_star, index=tickers).round(4))

print("\nNudged weights:")
print(pd.Series(w_nudged, index=tickers).round(4))

print("\nWeight change:")
print(pd.Series(weight_change, index=tickers).round(4))

print("\nTotal weight displacement:",
      round(np.linalg.norm(weight_change), 4))

print("\nOriginal sum:", round(w_star.sum(), 6))
print("Nudged sum:", round(w_nudged.sum(), 6))

Original weights:
SPY    -27.9142
IVV      8.2189
VOO     39.9820
QQQ    -12.2952
VTI     -9.8008
IWM     -1.9377
XLK      1.4479
SOXX     3.2989
dtype: float64

Nudged weights:
SPY    -14.7596
IVV      3.6965
VOO     31.9275
QQQ    -12.4427
VTI    -10.3039
IWM     -1.9204
XLK      1.5216
SOXX     3.2810
dtype: float64

Weight change:
SPY     13.1546
IVV     -4.5225
VOO     -8.0546
QQQ     -0.1475
VTI     -0.5031
IWM      0.0173
XLK      0.0737
SOXX    -0.0179
dtype: float64

Total weight displacement: 16.0827

Original sum: 1.0
Nudged sum: 1.0


# Add ridge penalty

In [ ]:
etas = [0, 0.01, 0.05, 0.10, 0.50, 1.00]

I = np.eye(n)
ones = np.ones((n, 1))

results = []

In [ ]:
for eta in etas:

    # KKT system for fully-invested ridge MVO
    KKT_ridge = np.block([
        [lam * Sigma.values + eta * I, ones],
        [ones.T, np.zeros((1, 1))]
    ])

    # Original expected returns
    rhs = np.concatenate([
        mu,
        [1.0]
    ])

    solution = np.linalg.solve(KKT_ridge, rhs)
    w_before = solution[:n]

    # Nudge SPY by +5 bps
    mu_nudged = mu.copy()
    mu_nudged[0] += 0.0005

    rhs_nudged = np.concatenate([
        mu_nudged,
        [1.0]
    ])

    solution_nudged = np.linalg.solve(
        KKT_ridge,
        rhs_nudged
    )

    w_after = solution_nudged[:n]

    # Weight swing
    swing = np.linalg.norm(w_after - w_before)

    # Turnover
    turnover = 0.5 * np.abs(w_after - w_before).sum()

    results.append([
        eta,
        swing,
        turnover
    ])

ridge_results = pd.DataFrame(
    results,
    columns=[
        "eta",
        "weight_swing",
        "turnover"
    ]
)

print(ridge_results.round(4))

# Save ridge portfolio when eta = 1
if eta == 1.00:
    w_ridge = w_before.copy()

     eta  weight_swing  turnover
0   0.00       16.0827   13.2456
1   0.01        0.0434    0.0401
2   0.05        0.0089    0.0085
3   0.10        0.0045    0.0042
4   0.50        0.0009    0.0008
5   1.00        0.0005    0.0004
6   0.00       16.0827   13.2456
7   0.01        0.0434    0.0401
8   0.05        0.0089    0.0085
9   0.10        0.0045    0.0042
10  0.50        0.0009    0.0008
11  1.00        0.0005    0.0004


# Go factor neutral - Test the alpha

In [ ]:
# Annualised factor mean
factor_mean = F["MKT"].mean() * 252

# Annualised alpha
alpha = mu - B["MKT"].values * factor_mean

print("Annualised alpha:")
print(pd.Series(alpha, index=tickers).round(4))

Annualised alpha:
SPY     0.0113
IVV     0.0124
VOO     0.0124
QQQ     0.0175
VTI     0.0059
IWM    -0.0101
XLK     0.1021
SOXX    0.4505
dtype: float64


In [ ]:
# Solve the factor-neutral portfolio

In [ ]:
D_np = D.values
B_np = B.values
alpha_np = alpha
n = len(tickers)

# Constraints:
# 1. Fully invested: 1'w = 1
# 2. Factor neutral: B'w = 0

C = np.column_stack([
    np.ones(n),
    B_np
])

d = np.array([1.0, 0.0])

# KKT system
KKT = np.block([
    [lam * D_np, C],
    [C.T, np.zeros((2, 2))]
])

rhs = np.concatenate([
    alpha_np,
    d
])

solution = np.linalg.solve(KKT, rhs)

w_fn = solution[:n]

print("Factor-neutral portfolio:")
print(pd.Series(w_fn, index=tickers).round(4))

print("\nSum of weights:")
print(round(w_fn.sum(), 6))

print("\nFactor exposure Bᵀw:")
print((B_np.T @ w_fn).round(8))

from scipy.optimize import minimize

n = len(tickers)

def objective(w):
    return -(alpha @ w - (lam / 2) * w @ D_np @ w)

constraints = [
    {
        "type": "eq",
        "fun": lambda w: w.sum() - 1
    },
    {
        "type": "eq",
        "fun": lambda w: B_np.T @ w
    },
    {
        "type": "ineq",
        "fun": lambda w: 2.0 - np.abs(w).sum()
    }
]

w0 = np.ones(n) / n

result = minimize(
    objective,
    w0,
    method="SLSQP",
    constraints=constraints,
    options={"maxiter": 2000, "ftol": 1e-10}
)

w_fn = result.x

print("Factor-neutral constrained portfolio:")
print(pd.Series(w_fn, index=tickers).round(4))

print("\nSum of weights:", round(w_fn.sum(), 6))
print("Factor exposure:", (B_np.T @ w_fn).round(8))
print("Gross exposure:", round(np.abs(w_fn).sum(), 6))

Factor-neutral portfolio:
SPY     -6838.8003
IVV      1485.3756
VOO     10800.3962
QQQ     -3312.5811
VTI     -2767.5604
IWM      -543.1909
XLK       336.2889
SOXX      841.0720
dtype: float64

Sum of weights:
1.0

Factor exposure Bᵀw:
[-0.]
Factor-neutral constrained portfolio:
SPY     0.0000
IVV     0.0000
VOO     1.6169
QQQ     0.0000
VTI    -0.0000
IWM     0.0000
XLK    -0.0000
SOXX   -0.6169
dtype: float64

Sum of weights: 1.0
Factor exposure: [-0.]
Gross exposure: 2.233798


# Tracking error comparison

In [ ]:
# Hypothetical equal-weighted benchmark
b = np.ones(n) / n

In [ ]:
# Active weights
a_naive = w_star - b
a_ridge = w_ridge - b
a_fn = w_fn - b

# Tracking error squared
TE2_naive = a_naive @ Sigma.values @ a_naive
TE2_ridge = a_ridge @ Sigma.values @ a_ridge
TE2_fn = a_fn @ Sigma.values @ a_fn

# Annualised tracking error
TE_naive = np.sqrt(TE2_naive)
TE_ridge = np.sqrt(TE2_ridge)
TE_fn = np.sqrt(TE2_fn)

# Comparison
TE_results = pd.DataFrame({
    "Portfolio": [
        "Naive MVO",
        "Ridge MVO (eta=1)",
        "Factor Neutral"
    ],
    "TE²": [
        TE2_naive,
        TE2_ridge,
        TE2_fn
    ],
    "Tracking Error": [
        TE_naive,
        TE_ridge,
        TE_fn
    ]
})

print(TE_results.round(6))

           Portfolio       TE²  Tracking Error
0          Naive MVO  0.379782        0.616264
1  Ridge MVO (eta=1)  0.010165        0.100823
2     Factor Neutral  0.098631        0.314056
